# X6 Label Generation on Google Colab

This notebook generates channel detection labels for the X6 trading model using Google Colab's resources.

**Features:**
- Parallel processing with multiple CPU cores
- Progress monitoring
- Automatic results saving to Google Drive
- Memory usage tracking

**Setup Required:**
1. Zip your x6 folder locally (including data)
2. Upload the zip to Colab
3. Run the cells below

**Estimated Runtime:** ~1-4 hours depending on step size

## Step 1: Check System Resources

In [ ]:
# Check available resources
!echo "CPU Cores:"
!nproc
!echo "\nRAM:"
!free -h
!echo "\nGPU (if available):"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU available (not needed for label generation)"

import psutil
print(f"\nPython RAM usage: {psutil.virtual_memory().percent:.1f}%")
print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.1f} GB")

# Typical Colab Free: 2 cores, 12-13 GB RAM
# Colab Pro: 4-8 cores, 25-50 GB RAM

## Step 2: Mount Google Drive (for saving results)

Your generated labels will be automatically saved here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create results directory
!mkdir -p /content/drive/MyDrive/x6_results
print("✓ Drive mounted successfully")
print("✓ Results will be saved to: /content/drive/MyDrive/x6_results/")

## Step 3: Upload Your x6.zip File

**Before running this cell, make sure you've created x6.zip locally:**

```bash
# On your Mac, run this in Terminal:
cd /Users/frank/Desktop/CodingProjects
zip -r x6.zip x6/ -x "x6/myenv/*" -x "x6/__pycache__/*" -x "x6/.git/*" -x "x6/checkpoints/*" -x "x6/*.pyc"
```

This creates a zip with:
- ✅ All v7/ code
- ✅ data/ folder (TSLA, SPY, VIX CSV files)
- ✅ train.py and other scripts
- ❌ myenv/ (excluded - will reinstall packages)
- ❌ .git/ (excluded - not needed)
- ❌ checkpoints/ (excluded - old models)

In [ ]:
from google.colab import files
import zipfile
import os

print("Please upload your x6.zip file...")
print("(This may take 2-5 minutes depending on file size)")
uploaded = files.upload()

# Extract the zip
zip_filename = list(uploaded.keys())[0]
print(f"\nExtracting {zip_filename}...")
!unzip -q {zip_filename}

# Change to x6 directory
%cd x6

print("\n✓ Files extracted successfully")
print("\nContents:")
!ls -lh

## Step 4: Verify Data Files

In [ ]:
# Check that data directory exists with required files
import os
from pathlib import Path

data_dir = Path('data')
required_files = ['TSLA_1min.csv']  # Minimum required
optional_files = ['SPY_1min.csv', 'VIX_History.csv']

print("Checking data files...")
print("="*50)

all_good = True
for filename in required_files:
    filepath = data_dir / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / (1024**2)
        print(f"✓ {filename}: {size_mb:.1f} MB")
    else:
        print(f"❌ {filename}: MISSING (required!)")
        all_good = False

for filename in optional_files:
    filepath = data_dir / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / (1024**2)
        print(f"✓ {filename}: {size_mb:.1f} MB")
    else:
        print(f"⚠️  {filename}: Not found (optional, will be fetched if needed)")

print("="*50)
if all_good:
    print("✓ All required data files present")
else:
    print("❌ Missing required files - please check your zip")

## Step 5: Install Dependencies

In [ ]:
%%capture install_output
# Install required packages (suppress output to keep notebook clean)
!pip install -q pandas numpy torch scipy tqdm openpyxl

# Verify installations
import pandas as pd
import numpy as np
import torch
from scipy import stats
from tqdm import tqdm

print("✓ All dependencies installed")
print(f"  - pandas: {pd.__version__}")
print(f"  - numpy: {np.__version__}")
print(f"  - torch: {torch.__version__}")
print(f"  - scipy: {stats.__version__ if hasattr(stats, '__version__') else 'installed'}")

## Step 6: Configure Label Generation Parameters

**Key Parameters to Adjust:**

- **`step`**: Spacing between samples
  - `500` = ~150 samples, ~2-3 hours
  - `1000` = ~75 samples, ~1-2 hours
  - `2000` = ~40 samples, ~30-60 min (good for testing)

- **`max_workers`**: Number of parallel processes
  - `None` = Auto-detect (recommended)
  - Colab Free: Usually 2 cores
  - Colab Pro: Usually 4-8 cores

In [ ]:
# Configuration
from pathlib import Path

CONFIG = {
    # Directories
    'data_dir': Path('data'),
    'cache_dir': Path('data/feature_cache'),
    
    # Scanning parameters
    'window': 50,              # Minimum warmup bars (don't change)
    'step': 500,               # ← ADJUST THIS: Sample spacing (500=more samples, 2000=fewer/faster)
    'min_cycles': 1,           # Minimum channel cycles required
    
    # Label generation parameters
    'max_scan': 100,           # Forward scan distance for labels
    'return_threshold': 20,    # Base return threshold (scaled per TF)
    'lookforward_bars': 200,   # Forward bars for exit tracking
    
    # Custom return thresholds (optional - use defaults if None)
    'custom_return_thresholds': None,  # e.g., {'daily': 7, 'weekly': 3}
    
    # Processing
    'parallel': True,          # Use parallel processing (recommended)
    'max_workers': None,       # ← ADJUST THIS: None=auto, or set number (2, 4, 8)
    'force_rebuild': True,     # Always rebuild from scratch
}

# Create cache directory
CONFIG['cache_dir'].mkdir(parents=True, exist_ok=True)

print("Configuration:")
print("="*50)
for key, value in CONFIG.items():
    if key in ['step', 'max_workers', 'parallel']:
        print(f"  {key}: {value}  ← Key parameter")
    else:
        print(f"  {key}: {value}")
print("="*50)

# Estimate samples
approx_samples = 440000 // CONFIG['step']  # Rough estimate
print(f"\nEstimated samples: ~{approx_samples}")
print(f"Estimated time: {approx_samples * 0.02:.0f}-{approx_samples * 0.04:.0f} minutes")
print(f"                ({approx_samples * 0.02 / 60:.1f}-{approx_samples * 0.04 / 60:.1f} hours)")

## Step 7: Run Label Generation

⚠️ **This cell will take 1-4 hours to complete**

**Tips while running:**
- Don't close the browser tab
- Keep Colab tab active (prevents disconnection)
- Monitor progress in the output below
- Use Step 11 (in another tab) to monitor system resources

In [ ]:
import time
import psutil
from datetime import datetime
from v7.training.dataset import prepare_dataset_from_scratch

print("="*60)
print("Starting Label Generation")
print("="*60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Parallel: {CONFIG['parallel']}")
print(f"Workers: {CONFIG['max_workers'] or 'Auto-detect'}")
print(f"Step size: {CONFIG['step']}")
print("="*60)
print("\nThis will take a while...")
print("Progress updates will appear below.\n")

start_time = time.time()
start_ram = psutil.virtual_memory().percent

try:
    # Run label generation
    train_samples, val_samples, test_samples = prepare_dataset_from_scratch(
        data_dir=CONFIG['data_dir'],
        cache_dir=CONFIG['cache_dir'],
        window=CONFIG['window'],
        step=CONFIG['step'],
        min_cycles=CONFIG['min_cycles'],
        max_scan=CONFIG['max_scan'],
        return_threshold=CONFIG['return_threshold'],
        lookforward_bars=CONFIG['lookforward_bars'],
        custom_return_thresholds=CONFIG.get('custom_return_thresholds'),
        force_rebuild=CONFIG['force_rebuild'],
        parallel=CONFIG['parallel'],
        max_workers=CONFIG['max_workers'],
    )
    
    elapsed_time = time.time() - start_time
    end_ram = psutil.virtual_memory().percent
    total_samples = len(train_samples) + len(val_samples) + len(test_samples)
    
    print("\n" + "="*60)
    print("✓ Label Generation Complete!")
    print("="*60)
    print(f"End time:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nTrain samples: {len(train_samples):4d}")
    print(f"Val samples:   {len(val_samples):4d}")
    print(f"Test samples:  {len(test_samples):4d}")
    print(f"Total:         {total_samples:4d}")
    print(f"\nTime elapsed:  {elapsed_time/60:.1f} minutes ({elapsed_time/3600:.2f} hours)")
    print(f"Time per sample: {elapsed_time/total_samples:.1f} seconds" if total_samples > 0 else "")
    print(f"RAM usage:     {start_ram:.1f}% → {end_ram:.1f}%")
    print("="*60)
    
except Exception as e:
    print(f"\n❌ Error during label generation: {e}")
    import traceback
    traceback.print_exc()
    print("\nTip: If out of memory, try increasing 'step' in CONFIG (e.g., from 500 to 1000)")

## Step 8: Inspect Generated Samples

In [ ]:
# Show detailed info about first sample
if train_samples:
    sample = train_samples[0]
    print("Sample 0 Details:")
    print("="*60)
    print(f"Timestamp: {sample.timestamp}")
    print(f"Channel direction: {sample.channel.direction}")
    print(f"Best window: {sample.best_window}")
    print(f"Bounce count: {sample.channel.bounce_count}")
    print(f"Complete cycles: {sample.channel.cycles}")
    print(f"R²: {sample.channel.r_squared:.3f}")
    print(f"Quality score: {sample.channel.quality_score:.3f}")
    
    print(f"\nLabels per timeframe (showing all 11):")
    print("="*60)
    for tf, labels in sample.labels.items():
        if labels:
            print(f"  {tf:8s}: duration={labels.duration_bars:3d} bars, "
                  f"break={'YES' if labels.permanent_break else 'NO ':3s}, "
                  f"valid={labels.duration_valid}")
        else:
            print(f"  {tf:8s}: No valid labels")
    
    # Show features info
    print(f"\nFeatures available: {sample.features is not None}")
    if hasattr(sample.features, 'tsla'):
        print(f"TSLA features: {len(sample.features.tsla)} timeframes")
        print(f"SPY features: {len(sample.features.spy)} timeframes")
        print(f"Cross-asset features: {len(sample.features.cross_containment)} timeframes")
else:
    print("No samples generated - check for errors above")

## Step 9: Save Results to Google Drive

**Results will be saved with timestamp to prevent overwrites**

In [ ]:
import shutil
from datetime import datetime
import json

# Create timestamped directory in Drive
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
total_samples = len(train_samples) + len(val_samples) + len(test_samples)
drive_dir = f"/content/drive/MyDrive/x6_results/samples_{total_samples}_{timestamp}"
!mkdir -p {drive_dir}

print(f"Saving results to: {drive_dir}")
print("="*60)

# Copy cached samples (pickle file)
cache_file = CONFIG['cache_dir'] / 'channel_samples.pkl'
if cache_file.exists():
    size_mb = cache_file.stat().st_size / (1024**2)
    shutil.copy(cache_file, f"{drive_dir}/channel_samples.pkl")
    print(f"✓ channel_samples.pkl ({size_mb:.1f} MB)")

# Copy metadata JSON if exists
json_file = CONFIG['cache_dir'] / 'channel_samples.json'
if json_file.exists():
    shutil.copy(json_file, f"{drive_dir}/channel_samples.json")
    print(f"✓ channel_samples.json (metadata)")

# Save config used for this run
config_save = {k: str(v) for k, v in CONFIG.items()}
config_save['total_samples'] = total_samples
config_save['train_samples'] = len(train_samples)
config_save['val_samples'] = len(val_samples)
config_save['test_samples'] = len(test_samples)
config_save['generation_time'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
config_save['elapsed_minutes'] = f"{elapsed_time/60:.1f}"

with open(f"{drive_dir}/config.json", 'w') as f:
    json.dump(config_save, f, indent=2)
print(f"✓ config.json (generation settings)")

# Create a README for easy reference
readme = f"""# Label Generation Results

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Samples
- Train: {len(train_samples)}
- Val:   {len(val_samples)}
- Test:  {len(test_samples)}
- Total: {total_samples}

## Settings
- Step size: {CONFIG['step']}
- Window: {CONFIG['window']}
- Parallel: {CONFIG['parallel']}
- Workers: {CONFIG['max_workers'] or 'Auto'}

## Time
- Elapsed: {elapsed_time/60:.1f} minutes ({elapsed_time/3600:.2f} hours)
- Per sample: {elapsed_time/total_samples:.1f} seconds

## Files
- channel_samples.pkl: Main data file (load with pickle)
- channel_samples.json: Metadata
- config.json: Generation settings

## Usage
Download channel_samples.pkl and place in your local:
/Users/frank/Desktop/CodingProjects/x6/data/feature_cache/
"""

with open(f"{drive_dir}/README.txt", 'w') as f:
    f.write(readme)
print(f"✓ README.txt")

print("="*60)
print(f"\n✓ All results saved to Google Drive!")
print(f"\nLocation: {drive_dir}")
print(f"\nYou can access these files anytime from Google Drive.")

## Step 10: Download Results to Local Machine

**Optional:** Download the pickle file directly to your computer

In [ ]:
from google.colab import files

cache_file = CONFIG['cache_dir'] / 'channel_samples.pkl'
if cache_file.exists():
    size_mb = cache_file.stat().st_size / (1024**2)
    print(f"Preparing to download channel_samples.pkl ({size_mb:.1f} MB)...")
    print("This may take a few minutes depending on file size.")
    files.download(str(cache_file))
    print("✓ Download started")
else:
    print("❌ Cache file not found")

## Step 11: Monitor System Resources

**Run this in parallel** while label generation is running (Step 7) to monitor CPU/RAM usage.

In [ ]:
# Run this cell while label generation is running to monitor resources
import time
import psutil
from IPython.display import clear_output

print("Starting resource monitor...")
print("Press the stop button (■) to stop monitoring\n")
time.sleep(2)

try:
    iteration = 0
    while True:
        clear_output(wait=True)
        
        ram = psutil.virtual_memory()
        cpu = psutil.cpu_percent(interval=1, percpu=True)
        cpu_avg = sum(cpu) / len(cpu)
        
        print("System Resource Monitor")
        print("="*60)
        print(f"CPU Cores: {len(cpu)}")
        for i, c in enumerate(cpu):
            bar = '█' * int(c / 5) + '░' * (20 - int(c / 5))
            print(f"  Core {i}: [{bar}] {c:5.1f}%")
        print(f"  Average: {cpu_avg:.1f}%")
        print()
        
        ram_bar = '█' * int(ram.percent / 5) + '░' * (20 - int(ram.percent / 5))
        print(f"RAM: [{ram_bar}] {ram.percent:.1f}%")
        print(f"  Used:      {ram.used/(1024**3):6.2f} GB")
        print(f"  Available: {ram.available/(1024**3):6.2f} GB")
        print(f"  Total:     {ram.total/(1024**3):6.2f} GB")
        print()
        
        if ram.percent > 90:
            print("⚠️  WARNING: RAM usage > 90% - may run out of memory!")
            print("   Consider increasing 'step' parameter for fewer samples.")
        elif ram.percent > 80:
            print("⚠️  High RAM usage - monitor closely")
        
        print(f"\nUpdate #{iteration + 1} - Refreshing every 5 seconds...")
        print("Press stop button (■) to exit monitor")
        
        iteration += 1
        time.sleep(5)
        
except KeyboardInterrupt:
    print("\n\n✓ Resource monitoring stopped")

## Troubleshooting

### ❌ Out of Memory Error?
**Solution:**
1. In Step 6, increase `'step'` parameter:
   - Change from `500` to `1000` (fewer samples)
   - Or change to `2000` (even fewer samples)
2. Reduce `'max_workers'` to `2` or `1`
3. Consider Colab Pro for more RAM (25-50 GB vs 12 GB)

### ⏱️ Taking Too Long?
**Normal times:**
- step=500: 2-3 hours (150 samples)
- step=1000: 1-2 hours (75 samples)
- step=2000: 30-60 min (40 samples)

**Speed up:**
- Increase `'step'` for fewer samples
- Verify `'parallel': True` is enabled
- Check CPU usage in Step 11 - should be >50%

### 🔌 Runtime Disconnected?
**Good news:** Results are saved to Drive!
1. Check `/content/drive/MyDrive/x6_results/`
2. If generation was incomplete, set `'force_rebuild': False` in Step 6
3. Re-run from Step 7 to resume

### 📊 Want More Samples?
**Decrease step size:**
- `step=250`: ~300 samples (4-6 hours)
- `step=100`: ~800 samples (10-15 hours - use Colab Pro!)

### 💰 Need More Power?
**Colab Pro benefits:**
- 25-50 GB RAM (vs 12 GB free)
- 4-8 CPU cores (vs 2 cores free)
- 24 hour runtime (vs 12 hour free)
- Background execution

### 📥 How to Use Generated Labels?
1. Download `channel_samples.pkl` from Drive or Step 10
2. Place in your local machine:
   ```
   /Users/frank/Desktop/CodingProjects/x6/data/feature_cache/channel_samples.pkl
   ```
3. Run training with:
   ```python
   ./myenv/bin/python train.py
   ```
4. The cache will be loaded automatically!